# Virelion-DCCP — FINAL One-Click Runtime Bug Hunt

Run the single code cell below in a fresh Colab runtime. It bootstraps its own checkout, installs validation dependencies only into `/content/dccp-bughunt-deps`, and runs the complete runtime/integration harness in a child process environment.

**It does not upgrade or replace Colab's global packages.**


In [ ]:
# One cell. No previous notebook state is required.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

ROOT = Path("/content/Virelion-DCCP-BUGHUNT")
DEPS = Path("/content/dccp-bughunt-deps")

# Remove only old test artifacts. The runner is created AFTER the checkout.
for p in (ROOT, DEPS):
    if p.exists():
        shutil.rmtree(p)

# Install validation packages into an isolated target directory.
DEPS.mkdir(parents=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--disable-pip-version-check",
    "--no-warn-script-location",
    "--target", str(DEPS),
    "jsonschema", "pytest", "pytest-cov", "coverage",
    "hypothesis", "ruff==0.16.7"
], check=True)

# Clone after dependency setup.
subprocess.run([
    "git", "clone", "--branch", "main", "--depth", "1",
    "https://github.com/Virelion-Biotech/Virelion-DCCP.git",
    str(ROOT)
], check=True)

runner = ROOT / "_runtime_bughunt.py"
runner_code = "from pathlib import Path\nimport json\nimport subprocess\nimport sys\nimport tempfile\n\nROOT = Path.cwd()\nSRC = ROOT / \"src\"\nSCENARIOS = ROOT / \"scenarios\"\n\nresults = []\n\ndef record(name, ok, detail=\"\"):\n    results.append({\"test\": name, \"ok\": bool(ok), \"detail\": detail})\n    print((\"PASS\" if ok else \"FAIL\") + \" :: \" + name)\n    if detail:\n        print(detail)\n\n# A. Package identity and every module import.\ntry:\n    import dccp\n    record(\n        \"import dccp from checkout\",\n        str(SRC) in str(dccp.__file__),\n        str(dccp.__file__),\n    )\nexcept Exception as exc:\n    record(\"import dccp from checkout\", False, f\"{type(exc).__name__}: {exc}\")\n\nfor path in sorted((SRC / \"dccp\").glob(\"*.py\")):\n    if path.name == \"__init__.py\":\n        continue\n    module = path.stem\n    try:\n        __import__(f\"dccp.{module}\")\n        record(f\"import dccp.{module}\", True)\n    except Exception as exc:\n        record(\n            f\"import dccp.{module}\",\n            False,\n            f\"{type(exc).__name__}: {exc}\",\n        )\n\n# B. Every scenario: load + audit.\ntry:\n    from dccp.scenario import load_scenario\n    from dccp.audit import audit_scenario\n\n    files = sorted(SCENARIOS.rglob(\"*.json\"))\n    record(\"scenario discovery\", bool(files), f\"count={len(files)}\")\n\n    for path in files:\n        try:\n            scenario = load_scenario(path)\n            audit = audit_scenario(scenario)\n            detail = \"\" if audit.passed else json.dumps({\n                \"schema_errors\": audit.schema_errors,\n                \"policy_errors\": audit.policy_errors,\n                \"warnings\": audit.policy_warnings,\n            })\n            record(\n                f\"audit {path.relative_to(ROOT)}\",\n                audit.passed,\n                detail,\n            )\n        except Exception as exc:\n            record(\n                f\"audit {path.relative_to(ROOT)}\",\n                False,\n                f\"{type(exc).__name__}: {exc}\",\n            )\nexcept Exception as exc:\n    record(\"scenario API setup\", False, f\"{type(exc).__name__}: {exc}\")\n\n# C. Registry/library/challenge/bundle integration.\ntry:\n    from dccp.registry import build_registry, write_registry\n    from dccp.library import (\n        discover_scenarios,\n        load_library,\n        materialize_challenge_set,\n        write_challenge_set,\n    )\n    from dccp.bundle import build_bundle\n\n    files = discover_scenarios(SCENARIOS)\n    with tempfile.TemporaryDirectory() as td:\n        tmp = Path(td)\n\n        registry = build_registry(\n            SCENARIOS,\n            exclude_paths=[tmp / \"registry.json\"],\n        )\n        payload = write_registry(\n            SCENARIOS,\n            tmp / \"registry.json\",\n        )\n        assert len(registry) == len(files) == payload[\"n_entries\"]\n\n        library = load_library(SCENARIOS)\n        assert len(library) == len(files)\n\n        challenge = materialize_challenge_set(SCENARIOS)\n        assert challenge[\"n_cases\"] == len(library)\n        assert isinstance(challenge[\"set_hash\"], str)\n        assert len(challenge[\"set_hash\"]) == 64\n\n        write_challenge_set(tmp / \"challenge.json\", SCENARIOS)\n        written = json.loads((tmp / \"challenge.json\").read_text())\n        assert written[\"set_hash\"] == challenge[\"set_hash\"]\n\n        bundle = build_bundle(\n            tmp / \"bundle\",\n            run_id=\"colab-bughunt\",\n            input_files=[files[0], files[0]],\n            producer_version=\"0.3.0\",\n            base_dir=ROOT,\n        )\n        assert len(bundle[\"inputs\"]) == 1\n        assert not bundle[\"inputs\"][0][\"path\"].startswith(\"/\")\n\n    record(\n        \"registry/library/challenge/bundle integration\",\n        True,\n        f\"files={len(files)}\",\n    )\nexcept Exception as exc:\n    record(\n        \"registry/library/challenge/bundle integration\",\n        False,\n        f\"{type(exc).__name__}: {exc}\",\n    )\n\n# D. Edge cases: non-finite values and outside-base rejection.\ntry:\n    from dccp.fingerprint import canonical_json\n    from dccp.bundle import build_bundle\n\n    for value in (float(\"nan\"), float(\"inf\"), float(\"-inf\")):\n        try:\n            canonical_json({\"x\": value})\n        except (ValueError, TypeError):\n            pass\n        else:\n            raise AssertionError(f\"canonical_json accepted {value!r}\")\n\n    with tempfile.TemporaryDirectory() as td:\n        root = Path(td)\n        base = root / \"base\"\n        base.mkdir()\n        outside = root / \"outside.txt\"\n        outside.write_text(\"x\")\n\n        try:\n            build_bundle(\n                base / \"bundle\",\n                run_id=\"outside\",\n                input_files=[outside],\n                base_dir=base,\n            )\n        except (ValueError, FileNotFoundError, RuntimeError):\n            pass\n        else:\n            raise AssertionError(\"accepted file outside base_dir\")\n\n    record(\"numeric/path edge cases\", True)\nexcept Exception as exc:\n    record(\n        \"numeric/path edge cases\",\n        False,\n        f\"{type(exc).__name__}: {exc}\",\n    )\n\n# E. Exact Ruff CI check and pytest.\nfor label, command in [\n    (\n        \"ruff 0.16.7 check src tests\",\n        [sys.executable, \"-m\", \"ruff\", \"check\", \"src\", \"tests\"],\n    ),\n    (\n        \"pytest -q\",\n        [sys.executable, \"-m\", \"pytest\", \"-q\"],\n    ),\n]:\n    p = subprocess.run(\n        command,\n        cwd=ROOT,\n        text=True,\n        capture_output=True,\n    )\n    detail = (p.stdout + \"\\n\" + p.stderr).strip()\n    record(\n        label,\n        p.returncode == 0,\n        \"\" if p.returncode == 0 else detail[-10000:],\n    )\n\n# F. CLI paths used by CI.\ncli_commands = [\n    [\"-m\", \"dccp.cli\", \"--version\"],\n    [\"-m\", \"dccp.cli\", \"--help\"],\n    [\n        \"-m\", \"dccp.cli\", \"validate\",\n        \"scenarios/examples/SCENARIO-001.ordinary-mi.json\",\n    ],\n    [\"-m\", \"dccp.cli\", \"audit-all\", \"scenarios/examples\"],\n    [\n        \"-m\", \"dccp.cli\", \"registry\",\n        \"--root\", \"scenarios/examples\",\n        \"--output\", \"/tmp/dccp-bughunt-registry.json\",\n    ],\n    [\n        \"-m\", \"dccp.cli\", \"release-gate\",\n        \"/tmp/dccp-bughunt-registry.json\",\n        \"--base\", \".\",\n    ],\n    [\n        \"-m\", \"dccp.cli\", \"materialize\",\n        \"--root\", \"scenarios/examples\",\n        \"--output\", \"/tmp/dccp-bughunt-challenge.json\",\n    ],\n    [\n        \"-m\", \"dccp.cli\", \"map-scores\",\n        \"--scores\", \"inflammatory=0.8,contractile_functional=0.4\",\n        \"--draft-id\", \"SCENARIO-902\",\n        \"-o\", \"/tmp/dccp-bughunt-draft.json\",\n    ],\n    [\n        \"-m\", \"dccp.cli\", \"validate\",\n        \"/tmp/dccp-bughunt-draft.json\",\n    ],\n    [\n        \"-m\", \"dccp.cli\", \"bundle\",\n        \"--output-dir\", \"/tmp/dccp-bughunt-bundle\",\n        \"--run-id\", \"ci\",\n        \"--input-files\",\n        \"scenarios/examples/SCENARIO-001.ordinary-mi.json\",\n    ],\n]\n\nfor command in cli_commands:\n    p = subprocess.run(\n        [sys.executable, *command],\n        cwd=ROOT,\n        env=dict(\n            **__import__(\"os\").environ,\n            PYTHONPATH=__import__(\"os\").pathsep.join([\n                str(ROOT / \"src\"),\n                str(Path(__import__(\"os\").environ.get(\"DEPS\", \"\"))),\n            ]),\n            PYTHONNOUSERSITE=\"1\",\n        ),\n        text=True,\n        capture_output=True,\n    )\n    detail = (p.stdout + \"\\n\" + p.stderr).strip()\n    record(\n        \"CLI \" + \" \".join(command),\n        p.returncode == 0,\n        \"\" if p.returncode == 0 else detail[-10000:],\n    )\n\n# G. Repeated deterministic library loading.\ntry:\n    from dccp.library import load_library\n    from dccp.provenance import canonical_hash\n\n    observed = []\n    for _ in range(25):\n        lib = load_library(SCENARIOS)\n        observed.append(canonical_hash({\n            \"ids\": sorted(x.scenario.scenario_id for x in lib),\n            \"digests\": sorted(x.digest for x in lib),\n        }))\n\n    record(\n        \"25 repeated library loads deterministic\",\n        len(set(observed)) == 1,\n    )\nexcept Exception as exc:\n    record(\n        \"25 repeated library loads deterministic\",\n        False,\n        f\"{type(exc).__name__}: {exc}\",\n    )\n\n# H. Public API resolution.\ntry:\n    import dccp\n\n    public = [name for name in dir(dccp) if not name.startswith(\"_\")]\n    unresolved = [\n        name for name in public\n        if getattr(dccp, name, None) is None\n    ]\n    record(\n        \"public API resolution\",\n        not unresolved,\n        f\"public={len(public)} unresolved={unresolved}\",\n    )\nexcept Exception as exc:\n    record(\n        \"public API resolution\",\n        False,\n        f\"{type(exc).__name__}: {exc}\",\n    )\n\npayload = {\n    \"commit\": subprocess.check_output(\n        [\"git\", \"rev-parse\", \"HEAD\"],\n        cwd=ROOT,\n        text=True,\n    ).strip(),\n    \"total\": len(results),\n    \"passed\": sum(item[\"ok\"] for item in results),\n    \"failed\": sum(not item[\"ok\"] for item in results),\n    \"results\": results,\n}\n(ROOT / \"colab_runtime_results.json\").write_text(\n    json.dumps(payload, indent=2) + \"\\n\",\n    encoding=\"utf-8\",\n)\n\nprint(\"\\n=== FINAL RESULT ===\")\nprint(json.dumps({\n    \"commit\": payload[\"commit\"],\n    \"total\": payload[\"total\"],\n    \"passed\": payload[\"passed\"],\n    \"failed\": payload[\"failed\"],\n}, indent=2))\n\nif payload[\"failed\"]:\n    print(\"\\n=== FAILURES ===\")\n    for item in results:\n        if not item[\"ok\"]:\n            print(\"\\nTEST:\", item[\"test\"])\n            print(item[\"detail\"])\n";
runner.write_text(runner_code, encoding="utf-8")

ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join([str(DEPS), str(ROOT / "src")])
ENV["PYTHONNOUSERSITE"] = "1"
ENV["DEPS"] = str(DEPS)

proc = subprocess.run(
    [sys.executable, str(runner)],
    cwd=ROOT,
    env=ENV,
    text=True,
    capture_output=True,
)

print(proc.stdout)
if proc.stderr:
    print("=== PROCESS STDERR ===
" + proc.stderr)

report_path = ROOT / "colab_runtime_results.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("
REPORT:")
    print(json.dumps({
        "commit": report["commit"],
        "total": report["total"],
        "passed": report["passed"],
        "failed": report["failed"],
    }, indent=2))
else:
    raise RuntimeError(
        "Runtime runner crashed before producing colab_runtime_results.json."
    )

print("
Runner exit code:", proc.returncode)
